# Kurzarbeitsbereitschaft

Dieses Notebook prüft rückblickend je abgeschlossenem Kalendermonat, ob die
Organisation die Voraussetzungen für Kurzarbeit erfüllt hätte: mindestens 30 % aller
Mitarbeitenden gelten als kurzarbeitsfähig, wenn ihr Anteil interner Arbeit
mindestens 24 % beträgt und ihr kumulierter Überstundenstand zum Monatsende unter
14 Stunden liegt.

**Vollständig unabhängig von der Umsatzprognose** des Hauptdashboards
(`01_dashboard.ipynb`) und den Bausteinen Bestand, Schulungsanmeldungen und Kosten -
ein Kapazitäts-/Personalsignal, kein Umsatz- oder Kostensignal. Es liest die Daten
nur; es verändert nichts.

**Für einzelne Personen bleibt unsichtbar, an welcher Bedingung sie scheitern** - alle
Ausgaben in diesem Notebook zeigen ausschließlich Aggregatzahlen.

**So wird es benutzt:** oben im Menü *Laufzeit → Alle ausführen*, dann von oben nach
unten lesen.

In [ ]:
# @title Umgebung einrichten und Kurzarbeit-Rohdaten laden

import importlib.util

if (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
):
    PAKET_REF = "main"
    PAKET_URL = f"git+https://github.com/it-agile/umsatzprognose-clockodo.git@{PAKET_REF}"
    SETUP_URL = (
        "https://raw.githubusercontent.com/it-agile/umsatzprognose-clockodo/"
        f"{PAKET_REF}/notebooks/setup.py"
    )
    !pip install --quiet "$PAKET_URL"
    !pip install --quiet --force-reinstall --no-deps "$PAKET_URL"
    !curl -sL "$SETUP_URL" -o setup.py

import setup

anzahl_monate = 6

rohdaten = setup.kurzarbeit_rohdaten(anzahl_monate=anzahl_monate)

## Rollenzuordnung und Schwellenwerte

Die Rollenzuordnung (wer laut Geschäftsführung/Vertrieb nie in den Zähler
kurzarbeitsfähiger Personen eingeht) ist eine personenbezogene Angabe und steht
deshalb in keiner Datei dieses Repositories - sie wird zur Laufzeit aus der
Umgebungsvariable/dem Colab-Secret `KURZARBEIT_ROLLENZUORDNUNG` gelesen.

Die drei Schwellenwerte sind unten frei änderbar, um Was-wäre-wenn-Szenarien
("ab welcher Schwelle wäre die Organisation vorbereitet gewesen") ohne Neuabruf
durchzuspielen - ein erneutes Ausführen dieser und der folgenden Zelle reicht.

In [ ]:
# @title Rollenzuordnung und Schwellenwerte konfigurieren

from umsatzprognose.clockodo import rollenzuordnung_automatisch
from umsatzprognose.domaene import Schwellenwerte

rollenzuordnung = rollenzuordnung_automatisch()

# Standardwerte - hier änderbar.
schwellenwerte = Schwellenwerte(
    anteil_interne_arbeit=0.24,
    ueberstunden_stunden=14.0,
    quote_organisation=0.30,
)

## Kurzarbeitsbereitschaft je Monat

Die Gesamtzahl einbezogener Personen als größerer Balken, die tatsächlich
kurzarbeitsfähigen als schmalerer Balken davor, dazu die Quote als Linie auf einer
zweiten y-Achse - sandbraun, wenn die Schwelle erreicht wurde, sonst violett (bewusst
keine Grün/Rot-Wertung: "erreicht" ist hier kein gutes Ergebnis). Werte per
Mauszeiger über Balken/Punkten. Darunter dieselben Zahlen als Tabelle.

In [ ]:
# @title Kurzarbeitsbereitschaft je Monat

from umsatzprognose.darstellung import diagramme
from umsatzprognose.domaene import bewertungen

ergebnisse = bewertungen(rohdaten, rollenzuordnung=rollenzuordnung, schwellenwerte=schwellenwerte)

diagramme.kurzarbeit_grafik(ergebnisse)

## Kurzarbeitsbereitschaft je Monat (Tabelle)

Der Standardmonat ist der jüngste abgeschlossene Monat; darunter eine Tabelle über
alle geladenen zurückliegenden Monate. Ausschließlich Aggregatzahlen - Anzahl und
Anteil kurzarbeitsfähiger Personen, die Zähler für die einzelnen Scheiter-Gründe,
ausgeschlossene und nicht bestimmbare Personen, sowie die verwendeten
Schwellenwerte.

In [ ]:
# @title Kurzarbeitsbereitschaft je Monat (Tabelle)

from umsatzprognose.darstellung.kurzarbeit import kurzarbeit_bericht

print(kurzarbeit_bericht(ergebnisse))

## Was zu den Zahlen zu wissen ist

Ein Hinweis erscheint je Monat, wenn Personen ausgeschlossen, unklassifizierte
Stunden aufgetreten sind, oder Personen nicht bestimmbar waren - jeweils nur als
Zähler, ohne Namen oder Einzelwerte.

In [ ]:
# @title Hinweise zu den Zahlen

from umsatzprognose.darstellung.kurzarbeit import kurzarbeit_hinweise_bericht

print(kurzarbeit_hinweise_bericht(ergebnisse))